# 📖 Capítol 4 - Algorismes i Text

Una seqüència genètica és una cadena (string) formada per caràcters d'un alfabet de quatre lletres: A, T, G, C, anomenats **bases**, que corresponen a les macromolècules de l'**ADN**. Un **gen** és una seqüència ordenada de bases i el **genoma** és la concatenació de tots els gens.

Cada cèl·lula produïda pel cos rep una còpia del genoma, però sovint aquesta còpia és alterada. Les possibles alteracions que es poden produir són, entre d'altres, la substitució d'una base per una altra o la pèrdua d'una base.

### ✍️ Exercici 1. Funció dna 

Fes una funció, anomenada "dna", basada en l'algorisme de Levensthein, que busqui dins d'una seqüència genètica una cadena genètica passada per paràmetre.

Aquesta funció ha de retornar la línia del fitxer on comença la cadena més semblant i la distància entre la cadena d'entrada i la cadena més semblant.

<span style="color:Blue">El càlcul  de la distància d'un patró al *substring* més semblant d'un text es pot fer amb l'algorisme de Levenshtein. L'única diferència és que s'ha d'inicialitzar la primera fila amb zeros i que la distància d'edició serà el valor mínim de l'última fila de la matriu de costos. També has de tenir en compte els costos en la inicialització de la primera columna.</span>


La seqüència genètica que farem servir és la del cromosoma 2 humà (fitxer HUMAN-DNA.txt).

Les primeres línies d'aquest fitxer tenen aquesta forma:

CCCATCTCTTTCTCATTCCTTGGTTGAGAACACGAACTTCAGGACTTGCCTCACACTAGGGCCCATTCTT
TGTTTCCCAGAAAGAAGAGGCTCTCCACACAGAGTCCCATGTACACCAGGCTGTCAACAAACATGAATTG
AATGAAGGAGTGGATGGTTGGGTGGAAGTGATTTAAGAAATCCTAACTGGGGAATTTCACTGGAAACTTA

En programar aquesta funció, cal que tinguis en compte que, en aplicacions bioinformàtiques, els costos de les operacions d'edició són lleugerament diferents dels que hem vist fins ara:

+ Per a un salt o inserció (al patró o al text), el cost és 2
+ Per a una substitució, el cost és 1
+ Quan hi ha correspondència, el cost és 0.

Usa els següents patrons:

In [134]:
def format_table(table):
    print("Debug: ")

    for i in table:
        print(i)

In [135]:
def init_table(patro, text, insr = 2) -> list[int, int]:
    """
    Return a table n * m, which n = len(patro)+1, m = len(text)+1
    """
    length = len(text) + 1
    width = len(patro) + 1

    table = [[None] * length for i in range(width)]
    # print(f"Debug: Expected table size is {len(patro)+1} * {len(text)+1}, real size is {len(table)} * {len(table[0])}")

    for i in range(len(table)):
        table[i][0] = i * insr

    for i in range(len(table[0])):
        table[0][i] = 0

    return table

In [137]:
def levenstheinsmithwaterman(patro, text, dlt = 2, insr = 2, subs = 1):
    """
    Aquesta funció implementa l'algorisme de Levensthein amb la variació d'Smith-Waterman. 
    És a dir, inicialitza la primera fila de la matriu a zeros.
    
    Parameters
    ----------
    patro: string
    text: sting
    
    dlt: int (default)
    insr: int (default)
    subs: int (default)
        Costos d'edició
        
    Returns
    -------
    minDistance: int  # Atenció: la distància mínima ja no serà l'extrem de la matriu, cal pensar què serà!
    """
    table = init_table(patro, text)  # returns a table n * m, which n = len(patro)+1, m = len(text)+1

    for n in range(1, len(table)):
        for m in range(1, len(table[0])):
            # print(f"[ {n}, {m}]", end = " ")
            cost = 1 if (text[m-1] != patro[n-1]) else 0
            table[n][m] = min(
                table[n][m-1] + insr,
                table[n-1][m] + dlt,
                table[n-1][m-1] + cost
            )

    # format_table(table)
    distancia_minima = min(table[-1])
    return distancia_minima



def dna(patro, fitxer = 'HUMAN-DNA.txt'):
    """
    Aquesta funció aplica l'algorisme de Levensthein amb la variació d'Smith Waterman 
    sobre una seqüència del dna per trobar diferents patrons.
    Treballa amb fitxers, i fa la cerca a cada línia.

    Parameters
    ----------
    patro: string
    fitxer: string (default)
    
    Returns
    -------
    linia: int
    distanciafinal: int
    """
    linia = 0
    distancia = len(patro) * 3  # an unreachable value

    with open(fitxer, "r") as file:
        for i, line in enumerate(file, start = 1):   # pending optimization: early return
            line.strip()

            curr_dist = levenstheinsmithwaterman(patro, line)
            if curr_dist < distancia:
                linia = i
                distancia = curr_dist

    print(linia, distancia)
    return (linia,distancia)

In [138]:
print(levenstheinsmithwaterman("123", "99991239999"))

0


In [139]:
assert dna('AGATACATTAGACAATAGAGATGTGGTC') == (32, 11)
assert dna('GTCAGTCTGGCCTTGCCATTGGTGCCACCA') == (352, 11)
assert dna('TACCGAGAAGCTGGATTACAGCATGTACCATCAT') == (233, 13)

32 11
352 11
233 13


Si a més de saber la distància volem saber quins canvis hi ha hagut haurem de modificar els anteriors algorismes per guardar els canvis a cada pas i un cop trobada la distància mínima desfer els passos i anar apuntant els canvis.

Recordem que hi pot haver 4 tipus de canvis

+ I: Insertion
+ D: Deletion
+ S: Substitution
+ C: Coincidence (no hi ha canvis)



### ✍️ Reescriu les anteriors funcions per registrar els canvis i per mostrar-los al final.

In [140]:
def init_tables(patro, text, insr = 2) -> tuple[list[list], list[list]]:
    length = len(text) + 1
    width = len(patro) + 1

    tableD = [[None] * length for i in range(width)]
    tableM = [[None] * length for i in range(width)]
    # print(f"Debug: Expected table size is {len(patro)+1} * {len(text)+1}, real size is {len(table)} * {len(table[0])}")

    """Init tableD"""
    for i in range(len(tableD)):
        tableD[i][0] = i * insr
    for i in range(len(tableD[0])):
        tableD[0][i] = 0

    """Init tableM"""
    for i in range(len(tableM)):
        tableM[i][0] = "I"
    for i in range(len(tableM[0])):
        tableM[0][i] = "EXCLUDE"  # the first row can't be used

    # format_table(tableD)
    # format_table(tableM)

    return (tableD, tableM)

In [141]:
def fill_matrices_in_place(patro, text, matD, matM, dlt = 2, insr = 2, subs = 1, coinc = 0) -> None:
    for n in range(1, len(matD)):
        for m in range(1, len(matD[0])):
            cs = coinc if (text[m-1] == patro[n-1]) else subs
            dltD, insrD, csD = matD[n][m-1] + insr, matD[n-1][m] + dlt, matD[n-1][m-1] + cs

            """fill distance"""
            minD = min(dltD, insrD, csD)
            matD[n][m] = minD

            """fill Movement"""
            if (minD == csD):
                matM[n][m] = "C" if (text[m-1] == patro[n-1]) else "S"
            elif (minD == dltD):
                matM[n][m] = "D"
            else:
                matM[n][m] = "I"
    
    # format_table(matD)
    # format_table(matM)

In [142]:
def make_matrices(patro, text) -> tuple[list[list], list[list]]:
    matriu_distancia, matriu_moviments = init_tables(patro, text)
    fill_matrices_in_place(patro, text, matriu_distancia, matriu_moviments)

    return matriu_distancia, matriu_moviments

In [143]:
"""
matD, matM = make_matrices("123", "999123999")
format_table(matD)
format_table(matM)
ini, fin, minD = search_ini_fin_min("123", "999123999", matD)
seq = make_seq_mov(matM, fin)
print(seq)
"""

matD, matM = make_matrices("123", "999")
format_table(matD)
format_table(matM)
ini, fin, minD = search_ini_fin_min("123", "999", matD)
print(ini, fin)
seq = make_seq_mov(matM, fin)
print(seq)

"""
matD, matM = make_matrices("999", "99999999")
format_table(matD)
format_table(matM)
ini, fin, minD = search_ini_fin_min("999", "99999999", matD)
seq = make_seq_mov(matM, fin)
print(seq)
"""


Debug: 
[0, 0, 0, 0]
[2, 1, 1, 1]
[4, 3, 2, 2]
[6, 5, 4, 3]
Debug: 
['EXCLUDE', 'EXCLUDE', 'EXCLUDE', 'EXCLUDE']
['I', 'S', 'S', 'S']
['I', 'S', 'S', 'S']
['I', 'S', 'S', 'S']
0 2
['I', 'S', 'S']


'\nmatD, matM = make_matrices("999", "99999999")\nformat_table(matD)\nformat_table(matM)\nini, fin, minD = search_ini_fin_min("999", "99999999", matD)\nseq = make_seq_mov(matM, fin)\nprint(seq)\n'

In [144]:
def search_ini_fin_min(patro, text, matD) -> tuple[int, int, int]:
    min_dist = min(matD[-1])

    fin_text = matD[-1].index(min_dist) - 1  # index in text = index in matrix - 1
    ini_text = fin_text - len(patro) + 1

    return ini_text, fin_text, min_dist

In [145]:
def make_seq_mov(matM, final) -> list[str]:
    i = len(matM)-1
    j = final + 1   # index in matrix = index in text + 1

    seq_mov = []
    while i >= 1 and j >= 0:
        mov = matM[i][j]
        seq_mov.append(mov)

        if mov == "C":
            i -= 1; j -= 1
        elif mov == "S":
            i -= 1; j -= 1
        elif mov == "I":
            j -= 1
        else:  # mov == "S"
            i -= 1
    
    seq_mov.reverse()
    return seq_mov

In [150]:
def levenstheinsmithwaterman(patro, text, dlt = 2, insr = 2, subs = 1):
    """
    Aquesta funció implementa l'algorisme de Levensthein amb la variació de Smith Waterman.
    Guarda a cada casella els canvis que hi ha hagut en una segona matriu de moviments
    
    Parameters
    ----------
    patro: string
    text: sting
    
    dlt: int (default)
    insr: int (default)
    subs: int (default)
        Costos d'edició
        
    Returns
    -------
    inici_text: posició inicial del text més semblant al patró
    final_text: posició final del text més semblant al patró
    distancia_minima: distancia entre el text i el patró
    matriu_moviments: matriu en la que s'indica C,S,D,I segons el moviment fet
    matriu_distancia: matriu amb les distancies
    sequencia_moviment: una llista amb la seqüència de canvis aplicats al patró per arribar al text
    """

    # fill tableD tableM
    matriu_distancia, matriu_moviments = make_matrices(patro, text)

    # get dist min and (ini, fin)
    inici_text, final_text, distancia_minima = search_ini_fin_min(patro, text, matriu_distancia)

    return (inici_text,final_text),distancia_minima,matriu_moviments



def dna(patro, fitxer = 'HUMAN-DNA.txt'):
    """
    Aquesta funció aplica l'algorisme de Levensthein amb la variació de Smith-Waterman sobre una seqüència del dna per trobar diferents patrons.
    
    Parameters
    ----------
    patro: string
    fitxer: string (default)
    
    Returns
    -------
    linia: linia on apareix el patró
    inici_text: posició inicial del text més semblant al patró
    final_text: posició final del text més semblant al patró
    distancia_minima: distancia entre el text i el patró
    matriu_moviments: matriu en la que s'indica C,S,D,I segons el moviment fet
    matriu_distancia: matriu amb les distancies
    sequencia_moviment: una llista amb la seqüència de canvis aplicats al patró per arribar al text
    """
    linia = None
    inici_text, final_text, sequencia_moviments = None, None, None
    distancia_minima = len(patro) * 3  # an unreachable value

    with open(fitxer, "r") as file:
        for i, line in enumerate(file, start = 1):   # pending optimization: early return
            line.strip()

            (curr_ini, curr_fin), curr_dist, curr_matM = levenstheinsmithwaterman(patro, line)
            if curr_dist < distancia_minima:
                linia = i
                distancia_minima = curr_dist
                inici_text, final_text = curr_ini, curr_fin

                # backtrack for seq mov
                sequencia_moviments = make_seq_mov(curr_matM, final_text)
    
    return (linia,(inici_text,final_text),distancia_minima, sequencia_moviments)

In [151]:
assert dna("CTGGTACCAGCTGTATTAGC") == (729,(11, 30), 6, ['C',  'C',  'C',  'C',  'C',  'C',  'C',  'S',  'C',  'S',  'C',  'S',  'S',  'S',  'C',  'C',  'S',  'C',  'C',  'C'])
assert dna("TCGTCATAAACCGCTGTGCC") == (213,(12, 31), 7, ['S',  'C',  'S',  'C',  'C',  'C',  'C',  'C',  'C',  'C',  'C',  'C',  'S',  'S',  'C',  'C',  'S',  'S',  'C',  'S'])
assert dna("TATACAAACGGAGTAGCTGT") == (286, (5, 24), 6, ['C',  'C',  'C',  'C',  'S',  'C',  'S',  'C',  'C',  'S',  'C',  'S',  'S',  'C',  'C',  'C',  'S',  'C',  'C',  'C'])
assert dna("AGGCGTAAGTCTTACGTATA") == (6, (41, 60), 7, ['C',  'S',  'C',  'S',  'S',  'C',  'C',  'C',  'C',  'C',  'C',  'S',  'C',  'S',  'S',  'C',  'S',  'C',  'C',  'C'])
assert dna("AACGGCATAGCCTGCAAGAG") == (434, (41, 60), 5, ['C',  'C',  'S',  'C',  'C',  'C',  'C',  'S',  'C',  'S',  'C',  'C',  'C',  'C',  'C',  'C',  'C',  'S',  'C',  'S'])
